# **Tutorial:** Introduction to Large Language Models (LLMs) with HuggingFace

## Objective:
The goal of this workshop is to introduce students to the basic concepts of large language models (LLMs) and give them hands-on experience using Hugging Face libraries to work with pretrained natural language models.

## Contents:
1. Introduction to Large Language Models (LLMs)
2. Environment setup
3. Practical examples
4. Exercise

## 1. Introduction to Large Language Models (LLMs)

Large language models (LLMs) are AI models trained on huge amounts of text to understand and generate natural language. These models can perform a variety of natural language processing (NLP) tasks such as translation, summarization, text generation, question answering, and more.

## 2. Environment setup

For this workshop, we'll use the Hugging Face `transformers` library. First, we need to install the required packages.

In [ ]:
# Install the transformers and torch libraries
%pip install transformers
%pip install torch

We import `pipeline` from the `transformers` library

In [10]:
# Avoids kernel crashes from duplicate OpenMP runtimes (common on macOS)
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# This environment also has TensorFlow installed; force transformers to use only PyTorch
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

from transformers import pipeline

## 3. Practical examples of loading and using pretrained models

### 3.1 Text generation (`text-generation`)

Let's load a pretrained Hugging Face model and use it to generate text.
The model weights around 1GB, so it may take a few minutes to download and load the model into memory.

In [11]:
# Load a text-generation pipeline with a small, current instruct model
generator = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B-Instruct')

Device set to use mps:0


In [12]:
# Instruct-tuned models expect a "messages" format (like a chat)
messages = [
    {"role": "user", # The role of the user is to provide input or instructions to the model. Other roles include "assistant" (the model's responses, meaning the model's output e,g.: "I think the answer is...") and "system" (instructions for the model).
     "content": "Once upon a time there was a kid who had a bike. Continue the story in 2 lines."
     }
]

text = generator(
    messages,
    max_new_tokens=80, # Maximum length of the generated text.
    num_return_sequences=1,
    temperature=0.7, # This parameter controls the randomness of the responses. Lower values make the model more predictable, while higher values make it more creative.
    )

# The model's reply is the last generated message
print(text[0]['generated_text'][-1]['content'])

Once upon a time, there was a little boy named Timmy who loved to ride his bicycle around the park and play on the trails.


Now let's use the model to generate text from several prompts

In [14]:
# Example of text generation with different prompts
prompts = [
    "In the future, AI will",
    "The secret to happiness is",
    "The quick brown fox"
]

for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]
    text = generator(messages, max_new_tokens=60, num_return_sequences=1)
    print(f"Prompt: {prompt}")
    print(f"Generated Text: {text[0]['generated_text'][-1]['content']}\n")

Prompt: In the future, AI will
Generated Text: in the future, AI (Artificial Intelligence) will continue to evolve and advance at an unprecedented pace, transforming various industries and shaping our daily lives in profound ways. Here’s a speculative look into some of the potential advancements:

1. **Advanced Personalization**: AI will enable more precise and personalized recommendations

Prompt: The secret to happiness is
Generated Text: There's no one-size-fits-all answer to the question of how to achieve happiness. However, there are several key factors that can contribute to overall well-being and happiness:

1. **Positive Relationships**: Strong social connections with loved ones, friends, and family are crucial for emotional support and a sense

Prompt: The quick brown fox
Generated Text: The quick brown fox jumps over the lazy dog.

This is a classic nursery rhyme in English that has been around for centuries. It's often used to teach children about the importance of being pun

> **Note:** *gated* models (Llama, Mistral) require `huggingface-cli login`. The ones we use here are public.

### 3.2 Question answering (`question-answering`)

We can use a question-answering pipeline to find answers within a given context.

In [15]:
# Load a question-answering pipeline
qa_pipeline = pipeline('question-answering')

# Define the context and the question
#context = "You are a mathematics teacher that can explain complex concepts in simple terms."
context = (
    "Artificial intelligence (AI) is a branch of computer science that aims to create machines that can perform tasks that would normally require human intelligence. "
    "Machine learning (ML) is a subset of AI that involves the use of algorithms and statistical models to enable computers to improve their performance on a task through experience. "
    "One common application of ML is in the field of natural language processing (NLP), where algorithms are used to understand and generate human language. "
    "For example, GPT-4o is a state-of-the-art language model developed by OpenAI that can generate human-like text based on a given prompt."
)
question = "What is machine learning?"

# Get the answer
result = qa_pipeline(question=question, context=context)
print(f"Question: {question}")
print(f"Answer: {result['answer']}")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


Question: What is machine learning?
Answer: a subset of AI that involves the use of algorithms and statistical models


### 3.3 Text summarization (`summarization`)

Use a pretrained model to generate a summary of a long text.

In [16]:
# Load a summarization pipeline
summarizer = pipeline('summarization')

# Long text to summarize
long_text = (
"Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. The term can also apply to any machine that exhibits traits associated with a human mind, such as learning and problem-solving. The ideal characteristic of artificial intelligence is its ability to rationalize and take actions that have the best chance of achieving a specific goal. A subset of artificial intelligence is machine learning, which refers to the idea that computer systems can learn from data, identify patterns, and make decisions with minimal human intervention."
)

# Generate the summary
summary = summarizer(long_text, max_length=50, min_length=25)
print("Summary:")
print(summary[0]['summary_text'])

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


Summary:
 Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions . The ideal characteristic of artificial intelligence is its ability to rationalize and take actions that have the best chance of


### 3.4 Text translation (`translation`)

We use a pretrained English-to-Spanish translation model

In [17]:
%pip install sentencepiece
import sentencepiece

# Load a translation pipeline, specifying the model
translation_pipeline = pipeline('translation', model='Helsinki-NLP/opus-mt-en-es')

# Define the text to translate
text = "Persistent homology is a method for computing topological features of a space at different spatial resolutions. More persistent features are detected over a wide range of spatial scales and are deemed more likely to represent true features of the underlying space rather than artifacts of sampling, noise, or particular choice of parameters"

# Get the translation
result = translation_pipeline(text)
print(f"Original text: {text}")
print(f"Translation: {result[0]['translation_text']}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.8 MB/s  0:00:0036m-:--:--
Note: you may need to restart the kernel to use updated packages.


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

ValueError: This tokenizer cannot be instantiated. Please make sure you have `sentencepiece` installed in order to use this tokenizer.

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

### 3.4 Sentiment analysis (`sentiment-analysis`)

Now we'll use a sentiment analysis model to predict whether a text is positive or negative.

In [ ]:
# Load the sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis")

# Text to analyze sentiment
text = "The price of the NVIDIA stock will increase tomorrow."
#text = "Today I have a quiz for the course Artificial Intelligence."
#text = "A cat is running in green fields in the summer."

# Perform sentiment analysis
sentiment_result = sentiment_analyzer(text)

# Print the sentiment result
print("Sentiment Analysis Result:")
for result in sentiment_result:
    print(f"Label: {result['label']}, Score: {result['score']}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


Sentiment Analysis Result:
Label: POSITIVE, Score: 0.985397458076477


### 3.5 Asking questions to a table (`table-question-answering`)

Now let's ask questions to a Pandas DataFrame

In [24]:
import pandas as pd

# Load a table-question-answering pipeline
#table_qa = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")
table_qa = pipeline("table-question-answering")

#Create a table as a pandas DataFrame
table = {
    "Name": ["Karen", "Anna", "Pedro"],
    "Age": ["22", "31", "19"],  # Ensure all entries are strings
    "Interest": ["Cybersecurity", "Numerical Analysis", "Functional Analysis"]
}

# Question about the table
#question= "What is the interest of Karen?"
#question = "What is the age of Pedro?"
question = "What is the average age of the table?" # 22 + 31 + 19 = 72 / 3 = 24

# Generate the answer
answer = table_qa(table=table, query=question)

# Print the question and the answer
print("Question:", question)
print("Answer:", answer['answer'])

No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19 (https://huggingface.co/google/tapas-base-finetuned-wtq).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


Question: What is the average age of the table?
Answer: AVERAGE > 22, 31, 19


## Other tasks:

- 'audio-classification'
- 'automatic-speech-recognition'
- 'conversational'
- 'depth-estimation'
- 'document-question-answering'
- 'feature-extraction'
- 'fill-mask'
- 'image-classification'
- 'image-feature-extraction'
- 'image-segmentation'
- 'image-to-image'
- 'image-to-text'
- 'mask-generation'
- 'ner', 'object-detection'
- 'question-answering'
- 'sentiment-analysis'
- 'summarization'
- 'table-question-answering'
- 'text-classification'
- 'text-generation'
- 'text-to-audio'
- 'text-to-speech'
- 'text2text-generation'
- 'token-classification'
- 'translation'
- 'video-classification'
- 'visual-question-answering'
- 'vqa'
- 'zero-shot-audio-classification'
- 'zero-shot-classification'
- 'zero-shot-image-classification'
- 'zero-shot-object-detection'
- 'translation_XX_to_YY'

### **Individual Activity (30 min):** Comparing Recent Hugging Face Models

Pick a task (`text-generation`, `zero-shot-classification`, `summarization`, `sentiment-analysis`, etc.) and two recent, lightweight models different from the ones used here. Suggestions: `Qwen/Qwen2.5-1.5B-Instruct`, `HuggingFaceTB/SmolLM2-360M-Instruct`, `microsoft/Phi-3-mini-4k-instruct`, or explore [Trending on Hugging Face](https://huggingface.co/models).

Try both with the same input and compare: Size and performance of each model. Which one answered better or faster? Save the notebook as a PDF and upload it to Moodle.

*Estimated time:* 30 min

In [ ]:
# Write your code below



**Save this Notebook as a PDF and upload it to the designated activity on Moodle.**  

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>